<a href="https://colab.research.google.com/github/Vuyo127/Movie-Recommendation-System/blob/Updated-system/Movie_Recommendation_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
movies = pd.read_csv("movies_metadata.csv", low_memory=False)
movies.head()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
print(os.listdir())

In [ ]:
##Load dataset
movies = pd.read_csv('movies_metadata.csv',
low_memory=False)


In [ ]:
movies.head()

In [ ]:
print("Number of rows:", movies.shape[0])
print("Number of columns:", movies.shape[1])

In [ ]:
movies.dtypes

In [ ]:
##Remove Duplicates
print("Duplicates before:" ,movies.duplicated().sum())
movies = movies.drop_duplicates()
print("Duplicates after:" ,movies.duplicated().sum())

In [ ]:
##Clean Movie IDs
movies["id"] = pd.to_numeric(
    movies["id"],
    errors="coerce"
)
movies = movies.dropna(subset=["id"]).copy()
movies["id"] = movies["id"].astype(int)


In [ ]:
movies = movies.drop_duplicates(subset="id")

In [ ]:
##Clean the movie titles
movies = movies.dropna(subset=["title"])

movies["title"] = movies["title"].astype(str).str.strip()

movies = movies[movies["title"] != ""]

In [ ]:
##Clean the release dates
movies["release_date"] = pd.to_datetime(
    movies["release_date"],
    errors="coerce"
)

movies["release_year"] = movies["release_date"].dt.year

In [ ]:
##Clean the numerival data
numeric_columns = [ "budget","revenue","runtime","popularity","vote_average","vote_count"
]

for column in numeric_columns:
    movies[column] = pd.to_numeric(
        movies[column],
        errors="coerce"
    )

In [ ]:
movies = movies[
    (movies["vote_average"] >= 0) &
    (movies["vote_average"] <= 10)
]

In [ ]:
movies = movies[
    movies["vote_count"].fillna(0) >= 0
]

In [ ]:
##Missing values
movies["budget"] = movies["budget"].replace(0, np.nan)
movies["revenue"] = movies["revenue"].replace(0, np.nan)
movies["runtime"] = movies["runtime"].replace(0, np.nan)

In [ ]:
movies["overview"] = movies["overview"].fillna("")
movies["original_language"] = (
    movies["original_language"].fillna("Unknown")
)
movies["original_title"] = (
    movies["original_title"].fillna(movies["title"])
)

In [ ]:
columns = ["id","title","original_title","overview","genres","original_language","release_date",
"release_year","runtime","popularity","vote_average","vote_count","budget","revenue"]

movies_clean = movies[columns].copy()

In [ ]:
##Sort dataset
movies_clean = movies_clean.sort_values(
    by=["release_year", "title"],
    ascending=[False, True]
)

In [ ]:
movies_clean = movies_clean.reset_index(drop=True)
movies_clean.head(20)

In [ ]:
print("Final rows:", movies_clean.shape[0])
print("Final columns:", movies_clean.shape[1])
print("Duplicate rows:",movies_clean.duplicated().sum())
print("Duplicate movie IDs:",movies_clean["id"].duplicated().sum())


In [ ]:
movies_clean.isnull().sum()

In [ ]:
##Remove columns with too many missing values
movies_clean = movies_clean.drop(columns=["budget", "revenue"]
)

In [ ]:
#Fill missing runtime with the median runtime
median_runtime = movies_clean["runtime"].median()
movies_clean["runtime"] = movies_clean["runtime"].fillna(median_runtime
)

In [ ]:
movies_clean.isnull().sum()

In [ ]:
movies_clean = movies_clean.sort_values(
    by=["release_year", "title"],
    ascending=[False, True],
    na_position="last"
)

movies_clean = movies_clean.reset_index(drop=True)

In [ ]:
print("Final rows:", movies_clean.shape[0])
print("Final columns:", movies_clean.shape[1])
print("Duplicate rows:", movies_clean.duplicated().sum())
print("Duplicate IDs:", movies_clean["id"].duplicated().sum())

In [ ]:
movies_clean.head(20)

##**Ratings.csv**

In [ ]:
ratings = pd.read_csv("ratings_small.csv")
ratings.head()

In [ ]:
ratings = pd.read_csv("ratings_small.csv")
print("Rows:", ratings.shape[0])
print("Columns:", ratings.shape[1])

ratings.head()

In [ ]:
ratings.shape

In [ ]:
ratings.info()

In [ ]:
##Check missing values
ratings.isnull().sum()

In [ ]:
##Clean userId and movieId

ratings["userId"] = pd.to_numeric(ratings["userId"],errors="coerce"
)

ratings["movieId"] = pd.to_numeric(ratings["movieId"],errors="coerce"
)

ratings = ratings.dropna(subset=["userId", "movieId"]).copy()

ratings["userId"] = ratings["userId"].astype(int)
ratings["movieId"] = ratings["movieId"].astype(int)

In [ ]:
# Check rating range
print("Minimum rating:", ratings["rating"].min())
print("Maximum rating:", ratings["rating"].max())
print("\nUnique rating values:")
print(sorted(ratings["rating"].unique()))

In [ ]:
#Check for invalid ratings

invalid_ratings = ratings[
    (ratings["rating"] < 0) |
    (ratings["rating"] > 5)
]

print("Invalid ratings:", len(invalid_ratings))

In [ ]:
# Check for duplicates
print("Duplicate rows:", ratings.duplicated().sum())

print("Duplicate user-movie ratings:",ratings.duplicated(
        subset=["userId", "movieId"]
    ).sum()
)

In [ ]:
# Convert timestamp to a readable date
ratings["timestamp"] = pd.to_datetime(
    ratings["timestamp"],
    unit="s",
    errors="coerce"
)

print(ratings.head())

In [ ]:
print("Missing timestamps:", ratings["timestamp"].isnull().sum())

In [ ]:
##Check whether rating movie IDs exist in the movie dataset

matched_movies = ratings["movieId"].isin(movies_clean["id"])
print("Matching movie IDs:", matched_movies.sum())
print("Unmatched movie IDs:", (~matched_movies).sum())

In [ ]:
unmatched_ids = ratings.loc[
    ~ratings["movieId"].isin(movies_clean["id"]),
    "movieId"]

print("Number of unmatched IDs:", len(unmatched_ids))
print("\nFirst 20 unmatched IDs:")
print(unmatched_ids.head(20).tolist())

In [ ]:
print("Smallest unmatched ID:", unmatched_ids.min())
print("Largest unmatched ID:", unmatched_ids.max())

In [ ]:
print("Movie IDs in movies_clean:")
print(movies_clean["id"].head(20).tolist())
print("\nMovie IDs in ratings:")
print(ratings["movieId"].head(20).tolist())

In [ ]:
print("Number of unique movie IDs in movies_clean:",
      movies_clean["id"].nunique())

print("Number of unique movie IDs in ratings:",
      ratings["movieId"].nunique())

In [ ]:
#Calculate movie ID matching percentage

matching_count = ratings["movieId"].isin(movies_clean["id"]).sum()
total_ratings = len(ratings)

match_percentage = (matching_count / total_ratings) * 100

print("Matching ratings:", matching_count)
print("Total ratings:", total_ratings)
print(f"Match percentage: {match_percentage:.2f}%")

In [ ]:
matching_movies = ratings.loc[
    ratings["movieId"].isin(movies_clean["id"]),
    "movieId"
].nunique()

unmatched_movies = ratings.loc[
    ~ratings["movieId"].isin(movies_clean["id"]),
    "movieId"
].nunique()

print("Unique matching movie IDs:", matching_movies)
print("Unique unmatched movie IDs:", unmatched_movies)

In [ ]:
links = pd.read_csv("links.csv")
print(links.shape)
links.head()

In [ ]:
# Clean the ID columns
links["movieId"] = pd.to_numeric(
    links["movieId"],
    errors="coerce"
)

links["tmdbId"] = pd.to_numeric(
    links["tmdbId"],
    errors="coerce"
)

links = links.dropna(
    subset=["movieId"]
).copy()

links["movieId"] = links["movieId"].astype(int)

In [ ]:
print("Missing TMDB IDs:", links["tmdbId"].isnull().sum())

In [ ]:
ratings_linked = ratings.merge(
    links[["movieId", "tmdbId"]],
    on="movieId",
    how="left"
)

print("Ratings after linking:", ratings_linked.shape)
ratings_linked.head()

In [ ]:
print("Ratings with TMDB ID:",ratings_linked["tmdbId"].notna().sum()
)

print("Ratings without TMDB ID:",ratings_linked["tmdbId"].isna().sum()
)

In [ ]:
ratings_movies = ratings_linked.merge(
    movies_clean,
    left_on="tmdbId",
    right_on="id",
    how="inner"
)

print("Final matched ratings:", len(ratings_movies))
print("Final columns:", ratings_movies.columns.tolist())

In [ ]:
print("Duplicate user-movie ratings:",ratings_movies.duplicated(subset=["userId", "movieId"]).sum()
)

In [ ]:
ratings_clean = ratings_movies[
    [
        "userId",
        "movieId",
        "rating",
        "timestamp",
        "id",
        "title"
    ]
].copy()

ratings_clean = ratings_clean.rename(
    columns={"id": "tmdbId"}
)

print("Final ratings dataset:", ratings_clean.shape)
ratings_clean.head()

In [ ]:
print("Missing values:")
print(ratings_clean.isnull().sum())

print("\nDuplicate rows:")
print(ratings_clean.duplicated().sum())

print("\nRating range:")
print(
    ratings_clean["rating"].min(),
    "to",
    ratings_clean["rating"].max()
)

In [ ]:
ratings_clean.to_csv(
    "ratings_cleaned.csv",
    index=False
)

print("ratings_cleaned.csv saved successfully.")

In [ ]:
import matplotlib.pyplot as plt
rating_counts = ratings_clean["rating"].value_counts().sort_index()
plt.figure(figsize=(8, 5))
plt.bar(
    rating_counts.index.astype(str),
    rating_counts.values
)
plt.xlabel("Rating")
plt.ylabel("Number of Ratings")
plt.title("Distribution of Movie Ratings")
plt.show()

In [ ]:
print("Average rating:", ratings_clean["rating"].mean())
print("Median rating:", ratings_clean["rating"].median())
print("Minimum rating:", ratings_clean["rating"].min())
print("Maximum rating:", ratings_clean["rating"].max())

In [ ]:
movie_rating_counts = (
    ratings_clean
    .groupby(["movieId", "title"])
    .size()
    .reset_index(name="rating_count")
    .sort_values(
        "rating_count",
        ascending=False
    )
)
movie_rating_counts.head(10)

In [ ]:
top_movies = movie_rating_counts.head(10)
plt.figure(figsize=(10, 6))
plt.barh(
    top_movies["title"].iloc[::-1],
    top_movies["rating_count"].iloc[::-1]
)
plt.xlabel("Number of Ratings")
plt.ylabel("Movie")
plt.title("Top 10 Most-Rated Movies")
plt.show()

In [ ]:
movie_average_ratings = (
    ratings_clean
    .groupby(["movieId", "title"])["rating"]
    .agg(
        average_rating="mean",
        rating_count="count"
    ).reset_index()
)

movie_average_ratings.head()

In [ ]:
highly_rated_movies = movie_average_ratings[
    movie_average_ratings["rating_count"] >= 20
].sort_values(
    "average_rating",
    ascending=False
)
highly_rated_movies.head(10)

In [ ]:
user_rating_counts = (
    ratings_clean
    .groupby("userId")
    .size()
    .reset_index(name="rating_count")
)
user_rating_counts.head()

In [ ]:
user_rating_counts = (
    ratings_clean
    .groupby("userId")
    .size()
    .reset_index(name="rating_count")
)
user_rating_counts.head()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(
    user_rating_counts["rating_count"],
    bins=30
)
plt.xlabel("Number of Ratings per User")
plt.ylabel("Number of Users")
plt.title("User Rating Activity")
plt.show()

In [ ]:
number_of_users = ratings_clean["userId"].nunique()
number_of_movies = ratings_clean["movieId"].nunique()
number_of_ratings = len(ratings_clean)
total_possible_ratings = (
    number_of_users * number_of_movies
)
sparsity = (

    (number_of_ratings / total_possible_ratings)
) * 100
print("Number of users:", number_of_users)
print("Number of movies:", number_of_movies)
print("Number of ratings:", number_of_ratings)
print(f"Sparsity: {sparsity:.2f}%")

In [ ]:
import ast

In [ ]:
def extract_genres(genre_data):
    try:
        genres = ast.literal_eval(genre_data)
        if isinstance(genres, list):
            return [
                genre["name"]
                for genre in genres
                if isinstance(genre, dict) and "name" in genre
            ]
        return []
    except:
        return []
movies_clean["genre_list"] = (
    movies_clean["genres"]
    .apply(extract_genres)
)
movies_clean[["title", "genre_list"]].head()

In [ ]:
from collections import Counter
genre_counts = Counter()
for genres in movies_clean["genre_list"]:
    genre_counts.update(genres)
genre_counts = pd.DataFrame(
    genre_counts.items(),
    columns=["genre", "movie_count"]
).sort_values(
    "movie_count",
    ascending=False
)
genre_counts.head(10)

In [ ]:
top_genres = genre_counts.head(10)
plt.figure(figsize=(10, 6))
plt.barh(
    top_genres["genre"].iloc[::-1],
    top_genres["movie_count"].iloc[::-1]
)
plt.xlabel("Number of Movies")
plt.ylabel("Genre")
plt.title("Top 10 Movie Genres")
plt.show()


In [ ]:
user_features = (
    ratings_clean
    .groupby("userId")["rating"]
    .agg(
        user_average_rating="mean",
        user_rating_count="count"
    )
    .reset_index()
)
user_features.head()

In [ ]:
#Task 5 and 6
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.preprocessing import normalize
from scipy.sparse import hstack

In [ ]:
def parse_genres(value):
  try:
    return [g["name"] for g in ast.literal_eval(value)]
  except:
    return []
cb = movies_clean.reset_index(drop=True).copy()
cb["genre_list"] = cb["genres"].apply(parse_genres)
cb["genre_text"] = cb["genre_list"].apply(
    lambda gs: " ".join(g.replace(" ", "") for g in gs)
)
print("Movies with no genres:", (cb["genre_list"].str.len() == 0).sum())
cb[["title", "genre_list"]].head()

In [ ]:
genre_tfidf = TfidfVectorizer(token_pattern=r"\S+")
overview_tfidf = TfidfVectorizer(stop_words="english", min_df=2, max_features=20000, sublinear_tf=True)
genre_matrix = genre_tfidf.fit_transform(cb["overview"])
overview_matrix = overview_tfidf.fit_transform(cb["overview"])
W_GENRE, W_OVERVIEW = 1.0, 1.0
content_matrix = normalize(
    hstack([W_GENRE * genre_matrix, W_OVERVIEW * overview_matrix]).tocsr()
)
print("Content matrix shape:", content_matrix.shape)

In [ ]:
cb["title_lower"] = cb["title"].str.lower().str.strip()
_lookup = cb.sort_values("vote_count", ascending=False).drop_duplicates("title_lower")
title_to_idx = pd.Series(_lookup.index, index=_lookup["title_lower"])

def recommend_content(title, n=10, min_votes=20):
    key = title.lower().strip()
    if key not in title_to_idx.index:
        close = cb.loc[cb["title_lower"].str.contains(key, regex=False), "title"].head(5).tolist()
        raise ValueError(f"'{title}' not found. Did you mean: {close}")

    idx = title_to_idx[key]
    sims = linear_kernel(content_matrix[idx], content_matrix).ravel()

    sims[idx] = -1                                    # exclude the movie itself
    sims[cb["vote_count"].fillna(0).to_numpy() < min_votes] = -1   # skip obscure titles

    top = np.argsort(-sims)[:n]
    out = cb.loc[top, ["title", "release_year", "genre_list"]].copy()
    out["similarity"] = sims[top].round(3)
    return out.reset_index(drop=True)

recommend_content("Toy Story")

In [ ]:
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split


r = ratings_linked.dropna(subset=["tmdbId"]).copy()
r["tmdbId"] = r["tmdbId"].astype(int)
r = r[r["tmdbId"].isin(movies_clean["id"])]


r = r.groupby(["userId", "tmdbId"], as_index=False)["rating"].mean()


MIN_MOVIE_RATINGS = 5
counts = r["tmdbId"].value_counts()
r = r[r["tmdbId"].isin(counts[counts >= MIN_MOVIE_RATINGS].index)].copy()


user_cat = pd.Categorical(r["userId"])
movie_cat = pd.Categorical(r["tmdbId"])
r["u"], r["m"] = user_cat.codes, movie_cat.codes
user_ids, movie_ids = user_cat.categories.to_numpy(), movie_cat.categories.to_numpy()
n_users, n_items = len(user_ids), len(movie_ids)

user_index = {uid: i for i, uid in enumerate(user_ids)}
movie_index = {mid: i for i, mid in enumerate(movie_ids)}
id_to_title = movies_clean.set_index("id")["title"]

sparsity = 1 - len(r) / (n_users * n_items)
print(f"{len(r):,} ratings | {n_users} users | {n_items} movies | {sparsity:.2%} empty")

In [ ]:
train, test = train_test_split(r, test_size=0.2, random_state=42)
global_mean = train["rating"].mean()

def build_matrices(df):
    """User-mean-centred ratings (dense) + a boolean 'was rated' mask + user means."""
    u, m = df["u"].to_numpy(), df["m"].to_numpy()
    sums = np.bincount(u, weights=df["rating"], minlength=n_users)
    cnts = np.bincount(u, minlength=n_users)
    user_mean = np.where(cnts > 0, sums / np.maximum(cnts, 1), global_mean)

    centered = np.zeros((n_users, n_items), dtype=np.float32)
    centered[u, m] = df["rating"].to_numpy() - user_mean[u]
    rated = np.zeros((n_users, n_items), dtype=bool)
    rated[u, m] = True
    return centered, rated, user_mean

R_train, rated_train, mean_train = build_matrices(train)

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))


print("RMSE  global mean :", round(rmse(test["rating"], global_mean), 4))
print("RMSE  user mean   :", round(rmse(test["rating"], mean_train[test["u"]]), 4))

In [ ]:
K = 30

def fit_item_knn(R_centered, k=K):

    nn = NearestNeighbors(n_neighbors=k + 1, metric="cosine", algorithm="brute")
    nn.fit(csr_matrix(R_centered.T))
    dist, idx = nn.kneighbors(csr_matrix(R_centered.T))
    return idx, 1 - dist                       # neighbour ids, similarities

nbr_idx, nbr_sim = fit_item_knn(R_train)

def predict_knn(u, i, R_centered, rated, user_mean):
    nbrs, sims = nbr_idx[i], nbr_sim[i]
    keep = (nbrs != i) & (sims > 0) & rated[u, nbrs]      # neighbours this user rated
    if not keep.any():
        return user_mean[u]
    pred = user_mean[u] + np.dot(sims[keep], R_centered[u, nbrs[keep]]) / sims[keep].sum()
    return float(np.clip(pred, 0.5, 5.0))

knn_preds = [predict_knn(u, i, R_train, rated_train, mean_train)
             for u, i in zip(test["u"], test["m"])]
print("RMSE  item k-NN   :", round(rmse(test["rating"], knn_preds), 4))

In [ ]:

N_FACTORS = 10

def fit_svd(R_centered, user_mean, k=N_FACTORS):
    U, s, Vt = svds(csr_matrix(R_centered), k=k)
    return (U * s) @ Vt + user_mean[:, None]

pred_svd = np.clip(fit_svd(R_train, mean_train), 0.5, 5.0)
svd_preds = pred_svd[test["u"].to_numpy(), test["m"].to_numpy()]
print("RMSE  SVD         :", round(rmse(test["rating"], svd_preds), 4))


for k in [5, 10, 20, 50]:
    p = np.clip(fit_svd(R_train, mean_train, k), 0.5, 5.0)
    print(f"  k={k:<3} RMSE:", round(rmse(test["rating"], p[test["u"], test["m"]]), 4))

In [ ]:

# Refit
R_all, rated_all, mean_all = build_matrices(r)
nbr_idx, nbr_sim = fit_item_knn(R_all)
pred_all = np.clip(fit_svd(R_all, mean_all), 0.5, 5.0)
movie_rating_counts = rated_all.sum(axis=0)

def similar_movies_cf(title, n=10):
    """Movies that the same users rated similarly (item-based k-NN)."""
    matches = movies_clean[movies_clean["title"].str.lower() == title.lower().strip()]
    matches = matches[matches["id"].isin(movie_index)]
    if matches.empty:
        raise ValueError(f"'{title}' has no ratings in the filtered data.")
    i = movie_index[matches.sort_values("vote_count", ascending=False)["id"].iloc[0]]
    keep = nbr_idx[i] != i
    ids = movie_ids[nbr_idx[i][keep]][:n]
    return pd.DataFrame({"title": id_to_title.loc[ids].to_numpy(),
                         "similarity": nbr_sim[i][keep][:n].round(3)})

def recommend_svd(user_id, n=10, min_ratings=10):
    """Top-n unseen movies by predicted rating for a user."""
    u = user_index[user_id]
    scores = pred_all[u].copy()
    scores[rated_all[u]] = -np.inf
    scores[movie_rating_counts < min_ratings] = -np.inf
    top = np.argsort(-scores)[:n]
    return pd.DataFrame({"title": id_to_title.loc[movie_ids[top]].to_numpy(),
                         "predicted_rating": scores[top].round(2)})

print(similar_movies_cf("Toy Story"))
print(recommend_svd(user_ids[0]))

In [ ]:
#Refit item-based k-NN using only training ratings. Needed because the
# Refit cell later overwrites nbr_idx/nbr_sim with a version trained on
nbr_idx_train, nbr_sim_train = fit_item_knn(R_train)

In [ ]:
tmdb_to_cb = pd.Series(cb.index, index=cb["id"])
tmdb_to_cb = tmdb_to_cb[~tmdb_to_cb.index.duplicated()]

cf_to_cb = tmdb_to_cb.reindex(movie_ids)
valid_content = cf_to_cb.notna().to_numpy()
cf_to_cb_idx = np.where(valid_content, cf_to_cb.fillna(-1).astype(int), -1)

content_cf = csr_matrix((n_items, content_matrix.shape[1]), dtype=np.float32)
if valid_content.any():
    content_cf = content_cf.tolil()
    content_cf[valid_content] = content_matrix[cf_to_cb_idx[valid_content]]
    content_cf = content_cf.tocsr()

print("Movies with content data:", valid_content.sum(), "/", n_items)

In [ ]:
# Create a content profile for each user from movies they rated 4 or higher,
# then calculate a content-based score for every movie for each user.

LIKE = 4.0

liked = train[train["rating"] >= LIKE]

L = csr_matrix(
    (
        np.ones(len(liked)),
        (liked["u"], liked["m"])
    ),
    shape=(n_users, n_items)
)

profiles = normalize(L @ content_cf)

content_scores = (
    profiles @ content_cf.T
).toarray()

print("Content scores shape:", content_scores.shape)

In [ ]:
# Predict every user's score for every movie using item similarity (k-NN),
# same logic as predict_knn but vectorized for speed.
rows = np.repeat(np.arange(n_items), nbr_idx_train.shape[1])
W = csr_matrix((nbr_sim_train.ravel(), (rows, nbr_idx_train.ravel())), shape=(n_items, n_items))
W.setdiag(0)
W.data[W.data < 0] = 0
W.eliminate_zeros()

num = np.asarray(W @ R_train.T).T
den = np.asarray(W @ rated_train.astype(np.float32).T).T
knn_scores = np.divide(num, den, out=np.zeros_like(num), where=den > 0)
print("k-NN score matrix:", knn_scores.shape)

In [ ]:
# Combine SVD and content scores into one hybrid score. Each is scaled
# to 0-1 per user first since they're on different numeric scales.
# alpha shifts weight toward SVD as a user accumulates more ratings.
def minmax_rows(A):
    A = A.astype(np.float32)
    lo = A.min(axis=1, keepdims=True)
    hi = A.max(axis=1, keepdims=True)
    return (A - lo) / np.maximum(hi - lo, 1e-9)

C = 20
n_train_ratings = rated_train.sum(axis=1)
alpha = (n_train_ratings / (n_train_ratings + C))[:, None]

hybrid = alpha * minmax_rows(pred_svd) + (1 - alpha) * minmax_rows(content_scores)
print("Hybrid matrix:", hybrid.shape)

In [ ]:
# Returns top-N movie indices for every user.
# Already-rated movies and movies with too few ratings are excluded.

item_counts = rated_train.sum(axis=0)

def top_n(S, n=10, min_ratings=10):
    S = S.astype(float).copy()
    S[rated_train] = -np.inf
    S[:, item_counts < min_ratings] = -np.inf
    top = np.argpartition(-S, n, axis=1)[:, :n]
    order = np.argsort(-np.take_along_axis(S, top, axis=1), axis=1)
    return np.take_along_axis(top, order, axis=1)

In [ ]:
liked_counts = np.bincount(liked["m"], minlength=n_items)

# Score table per model so they can be compared
score_tables = {
    "Popularity": np.tile(liked_counts, (n_users, 1)),
    "Content":    content_scores,
    "ItemKNN":    knn_scores,
    "SVD":        pred_svd,
    "Hybrid":     hybrid,
}
recs = {name: top_n(S) for name, S in score_tables.items()}

In [ ]:
#Task 8

In [ ]:
# Convert a user’s top-N indices into readable movie titles.
def show_recommendations(user_number, model="Hybrid", n=10):
    indices = recs[model][user_number][:n]
    ids = movie_ids[indices]
    df = pd.DataFrame({
        "Rank": range(1, n + 1),
        "Movie": id_to_title.reindex(ids).fillna("Unknown").to_numpy()
    })
    return df.set_index("Rank")

show_recommendations(0, model="Hybrid")

In [ ]:
# Measure how many of the top-10 recommendations actually appear in the user’s held-out test ratings.
good = test[test["rating"] >= LIKE]
relevant = good.groupby("u")["m"].apply(set).to_dict()   # answer key from test set

def precision_at_k(recs_matrix, k=10):
    hits, total = 0, 0
    for u, rel in relevant.items():
        if not rel:
            continue
        hits += len(set(recs_matrix[u, :k]) & rel)
        total += k
    return hits / total if total else np.nan

evaluation_df = pd.DataFrame([
    {"Model": name, "Precision@10": round(precision_at_k(R), 4)}
    for name, R in recs.items()
])
evaluation_df

In [ ]:
assert len(set(recs["SVD"][0]) & set(np.where(rated_train[0])[0])) == 0


print(id_to_title.reindex(movie_ids[recs["Hybrid"][0]]))
print((recs["Popularity"][0] == recs["Popularity"][1]).all())

In [ ]:
##Task 9

In [ ]:
##Precision/Recall/F1 for the content-based model
def evaluate_content_based(n_users=50, k=10):
    results = []
    users = ratings['userId'].unique()[:n_users]

    for user_id in users:
        train_liked = get_liked_movies(user_id, train_ratings)
        test_liked = get_liked_movies(user_id, test_ratings)

        if not train_liked or not test_liked:
            continue

        seed_movie = train_liked[0]
        recs = get_similar_movies(seed_movie, n=k)['title'].tolist()

        hits = [t for t in recs if t in test_liked]
        precision = len(hits) / len(recs) if recs else 0
        recall = len(hits) / len(test_liked)
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0

        results.append({'user_id': user_id, 'precision': precision, 'recall': recall, 'f1': f1})

    results_df = pd.DataFrame(results)
    print(f"\n[CONTENT-BASED] Evaluated {len(results_df)} users")
    print(f"[CONTENT-BASED] Average Precision@{k}: {results_df['precision'].mean():.4f}")
    print(f"[CONTENT-BASED] Average Recall@{k}: {results_df['recall'].mean():.4f}")
    print(f"[CONTENT-BASED] Average F1@{k}: {results_df['f1'].mean():.4f}")
    return results_df

content_results = evaluate_content_based()

In [ ]:
##Precision/Recall/F1 for the popularity baseline
def evaluate_popularity_baseline(n_users=50, k=10):
    results = []
    users = ratings['userId'].unique()[:n_users]
    top_k_popular = get_popular_movies(k)['title'].tolist()

    for user_id in users:
        test_liked = get_liked_movies(user_id, test_ratings)
        if not test_liked:
            continue

        hits = [t for t in top_k_popular if t in test_liked]
        precision = len(hits) / len(top_k_popular)
        recall = len(hits) / len(test_liked)
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0

        results.append({'user_id': user_id, 'precision': precision, 'recall': recall, 'f1': f1})

    results_df = pd.DataFrame(results)
    print(f"\n[BASELINE] Evaluated {len(results_df)} users")
    print(f"[BASELINE] Average Precision@{k}: {results_df['precision'].mean():.4f}")
    print(f"[BASELINE] Average Recall@{k}: {results_df['recall'].mean():.4f}")
    print(f"[BASELINE] Average F1@{k}: {results_df['f1'].mean():.4f}")
    return results_df

baseline_results = evaluate_popularity_baseline()

In [ ]:
##RMSE/MAE (only if a collaborative filtering model predicting ratings exists — skip if not)
from sklearn.metrics import mean_squared_error, mean_absolute_error

def evaluate_rating_predictions(predict_fn, test_df=test_ratings):
    # predict_fn(userId, movieId) -> predicted rating
    y_true, y_pred = [], []
    for _, row in test_df.iterrows():
        pred = predict_fn(row['userId'], row['movieId'])
        if pred is not None:
            y_true.append(row['rating'])
            y_pred.append(pred)

    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mae = mean_absolute_error(y_true, y_pred)
    print(f"[COLLABORATIVE FILTERING] RMSE: {rmse:.4f}")
    print(f"[COLLABORATIVE FILTERING] MAE: {mae:.4f}")
    return rmse, mae

# Example call once Person 2's model function exists:
# evaluate_rating_predictions(predict_rating_cf)

In [ ]:
##Final comparison table
comparison = pd.DataFrame({
    'Model': ['Popularity Baseline', 'Content-Based (Genre)'],
    'Precision@10': [baseline_results['precision'].mean(), content_results['precision'].mean()],
    'Recall@10': [baseline_results['recall'].mean(), content_results['recall'].mean()],
    'F1@10': [baseline_results['f1'].mean(), content_results['f1'].mean()],
})

print(comparison)